# Computer Vision & Image Processing



---

## 1. Learning Objectives

- Load, inspect, and manipulate images using Pillow (PIL).
- Apply color space conversions, resizing, and image augmentation.
- Build and train a CNN with TensorFlow/Keras on CIFAR-10.
- Preprocess face images using the LFW dataset (sklearn).
- Understand PyTorch tensors and autograd basics.

## Concept / Theory

### Images as Tensors
A digital image is a 3D array: **(Height × Width × Channels)**
- Grayscale: H × W × 1
- RGB color: H × W × 3
- Pixel values: 0-255 (uint8) or 0.0-1.0 (float32 for NN)

### Convolution Operation
A filter (kernel) slides across the image computing dot products. This detects local patterns (edges, textures) regardless of position.

### CNN Architecture
```
Input Image → [Conv2D+ReLU] → [MaxPool] → [Conv2D+ReLU] → [MaxPool]
→ Flatten → Dense → Output
```

### Why CNN over Dense for Images?
- **Parameter sharing:** Same filter reused across all positions
- **Local connectivity:** Each neuron sees only a local patch
- **Translation invariance:** Detects patterns anywhere in image

In [ ]:
# Setup
!pip install -q tensorflow torch torchvision pillow matplotlib scikit-learn numpy

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter, ImageDraw, ImageFont
import io, os, random
import tensorflow as tf
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.datasets import fetch_lfw_people

print('TensorFlow:', tf.__version__)
print('PyTorch:', torch.__version__)
print('CUDA (PyTorch):', torch.cuda.is_available())

## Practical Lab 1 – Image Loading & Manipulation with Pillow

### Objective
Create, load, inspect, and transform images programmatically.

### Explanation
Pillow (PIL) is the standard Python library for image manipulation. Before feeding images to a neural network, we must standardize size, format, and value range.

In [ ]:
# Lab 1: Create a synthetic test image (no file needed)
def make_gradient_image(w=200, h=150):
    img = Image.new('RGB', (w, h))
    pixels = img.load()
    for x in range(w):
        for y in range(h):
            r = int(255 * x / w)
            g = int(255 * y / h)
            b = 128
            pixels[x, y] = (r, g, b)
    return img

sample_img = make_gradient_image()
print('Image size (W×H):', sample_img.size)
print('Mode:', sample_img.mode)

plt.figure(figsize=(5, 4))
plt.imshow(sample_img)
plt.title('Sample RGB Image'); plt.axis('off'); plt.show()

In [ ]:
# Lab 1: Basic Pillow operations
# Resize
resized = sample_img.resize((64, 64))
print('Resized:', resized.size)

# Grayscale
gray = sample_img.convert('L')
print('Grayscale mode:', gray.mode)

# Flip and rotate
flipped  = ImageOps.mirror(sample_img)
rotated  = sample_img.rotate(45, expand=False)

# Convert to NumPy array (for ML)
img_arr = np.array(sample_img)
print('\nNumPy array shape:', img_arr.shape)    # (H, W, C)
print('Value range:', img_arr.min(), '-', img_arr.max())

# Normalize to [0,1]
img_norm = img_arr.astype('float32') / 255.0
print('Normalized range:', img_norm.min(), '-', img_norm.max())

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, im, title in zip(axes, [sample_img, gray, flipped, rotated], ['Original','Grayscale','Flipped','Rotated 45°']):
    ax.imshow(im, cmap='gray' if im.mode=='L' else None)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## Practical Lab 2 – Color Spaces, Resizing & Augmentation

### Objective
Apply common image preprocessing operations used before neural network training.

### Explanation
Data augmentation artificially increases dataset size by applying random transforms (flip, rotate, crop, brightness) during training, reducing overfitting.

In [ ]:
# Lab 2: Color space conversions
img_rgb   = sample_img                        # RGB
img_gray  = sample_img.convert('L')           # Grayscale
img_ycbcr = sample_img.convert('YCbCr')       # Luminance + Chroma (used in video)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, img, title in zip(axes, [img_rgb, img_gray, img_ycbcr], ['RGB','Grayscale','YCbCr']):
    ax.imshow(img, cmap='gray' if img.mode=='L' else None)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

# Channel splitting
R, G, B = sample_img.split()
print('Channel shapes (as arrays):', np.array(R).shape)

In [ ]:
# Lab 2: Image augmentation pipeline
def augment(pil_img):
    """Apply random augmentations (as done in ML training pipelines)."""
    # Random horizontal flip
    if random.random() > 0.5:
        pil_img = ImageOps.mirror(pil_img)
    # Random brightness
    from PIL import ImageEnhance
    factor = random.uniform(0.7, 1.3)
    pil_img = ImageEnhance.Brightness(pil_img).enhance(factor)
    # Random blur
    if random.random() > 0.7:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=1))
    return pil_img

random.seed(42)
augmented = [augment(make_gradient_image()) for _ in range(6)]
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, aug in zip(axes, augmented):
    ax.imshow(aug); ax.axis('off')
plt.suptitle('Augmented versions of the same image')
plt.tight_layout(); plt.show()

In [ ]:
# Lab 2: TorchVision transforms (used in PyTorch pipelines)
base_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),                        # PIL → Tensor [C,H,W], [0,1]
    transforms.Normalize([0.5]*3, [0.5]*3)        # normalize to [-1,1]
])
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
tensor = base_transform(sample_img)
print('Tensor shape (C,H,W):', tensor.shape)
print('Tensor value range:', tensor.min().item(), 'to', tensor.max().item())

## Practical Lab 3 – CNN for CIFAR-10 Image Classification

### Objective
Build and train a CNN to classify 10 types of objects in color images.

### Explanation
CIFAR-10 contains 60,000 32×32 color images in 10 classes (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck). CNNs are state-of-the-art for this type of task.

In [ ]:
# Lab 3: Load CIFAR-10
(X_tr, y_tr), (X_te, y_te) = tf.keras.datasets.cifar10.load_data()
X_tr = X_tr.astype('float32') / 255.0
X_te = X_te.astype('float32') / 255.0
y_tr = y_tr.squeeze(); y_te = y_te.squeeze()

class_names = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print(f'Train: {X_tr.shape} | Test: {X_te.shape}')

fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_tr[i])
    ax.set_title(class_names[y_tr[i]], fontsize=7)
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Images'); plt.tight_layout(); plt.show()

In [ ]:
# Build CNN model
cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    
    # Block 1
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.2),
    
    # Block 2
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.3),
    
    # Classifier head
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax')
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
# Train CNN (5 epochs for demo; increase for better accuracy)
cnn_history = cnn.fit(
    X_tr, y_tr,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)],
    verbose=1
)
cnn_loss, cnn_acc = cnn.evaluate(X_te, y_te, verbose=0)
print(f'\nCIFAR-10 CNN Test Accuracy: {cnn_acc:.4f}')
print('Note: With 20+ epochs, accuracy typically reaches ~75-80%')

In [ ]:
# Show predictions
y_pred_prob = cnn.predict(X_te[:12], verbose=0)
y_pred      = y_pred_prob.argmax(axis=1)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, img, true_lbl, pred_lbl in zip(axes.flat, X_te[:12], y_te[:12], y_pred):
    ax.imshow(img)
    color = 'green' if true_lbl == pred_lbl else 'red'
    ax.set_title(f'T:{class_names[true_lbl]}\nP:{class_names[pred_lbl]}', color=color, fontsize=7)
    ax.axis('off')
plt.suptitle('CIFAR-10 Predictions (green=correct, red=wrong)')
plt.tight_layout(); plt.show()

## Practical Lab 4 – Face Image Preprocessing with LFW

### Objective
Load real face images and apply preprocessing for deep learning.

### Explanation
LFW (Labeled Faces in the Wild) is a benchmark face dataset. We load it via sklearn, then apply standard face preprocessing steps.

In [ ]:
# Lab 4: Load LFW dataset
print('Loading LFW dataset (downloads ~30MB on first run)...')
lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=False)
print(f'LFW: {lfw.images.shape} images, {len(lfw.target_names)} people')

# Show sample faces
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(lfw.images[i], cmap='gray')
    ax.set_title(lfw.target_names[lfw.target[i]].split()[-1], fontsize=7)
    ax.axis('off')
plt.suptitle('LFW Face Dataset Samples')
plt.tight_layout(); plt.show()

In [ ]:
# Lab 4: Preprocessing pipeline for faces
sample_face = lfw.images[0]  # (H, W) float [0,1]
print('Original face shape:', sample_face.shape)
print('Value range:', sample_face.min().round(3), '-', sample_face.max().round(3))

# Convert to PIL
face_pil = Image.fromarray((sample_face * 255).astype(np.uint8))

# Resize to standard 48×48
face_48 = face_pil.resize((48, 48))

# Normalize and add batch dimension for NN input
face_tensor = np.array(face_48).astype('float32') / 255.0
face_tensor = face_tensor[np.newaxis, :, :, np.newaxis]  # (1, 48, 48, 1)
print('NN-ready tensor shape:', face_tensor.shape)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(face_pil, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(face_48, cmap='gray'); axes[1].set_title('Resized 48×48'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Practical Lab 5 – PyTorch Tensors & Autograd Basics

### Objective
Understand PyTorch's tensor operations and automatic differentiation.

### Explanation
PyTorch is the other major deep learning framework (alongside TensorFlow). Its key feature is **autograd** — automatic computation of gradients for any tensor operation.

In [ ]:
# Lab 5: PyTorch tensors
import torch

# Create tensors
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print('Tensor a:\n', a)
print('Shape:', a.shape)
print('Dtype:', a.dtype)
print('Device:', a.device)

# Operations
b = torch.ones(2, 3)
print('\na + b:', a + b)
print('a * 2:', a * 2)
print('Matrix multiply (a.T @ a):\n', (a.T @ a))

In [ ]:
# Lab 5: Autograd (automatic differentiation)
x = torch.tensor([3.0], requires_grad=True)  # track gradients
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

y = w * x + b   # y = 2x + 1 = 7
loss = (y - 5) ** 2  # MSE: (7-5)² = 4

# Backward pass: compute gradients
loss.backward()
print('x:', x.item(), '| w:', w.item(), '| b:', b.item())
print('y:', y.item(), '| loss:', loss.item())
print('\nGradients:')
print('  d(loss)/dw:', w.grad.item())   # chain rule
print('  d(loss)/db:', b.grad.item())
print('  d(loss)/dx:', x.grad.item())

In [ ]:
# Lab 5: Build a simple PyTorch model
import torch.nn as nn
import torch.optim as optim

# 2-class classifier
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Use Iris data
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
iris = load_iris()
X_i, y_i = iris.data[iris.target < 2], iris.target[iris.target < 2]  # binary
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_i, y_i, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_sc = torch.tensor(sc.fit_transform(X_tr_i), dtype=torch.float32)
X_te_sc = torch.tensor(sc.transform(X_te_i), dtype=torch.float32)
y_tr_t = torch.tensor(y_tr_i, dtype=torch.long)

model_pt = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.01)

for epoch in range(50):
    model_pt.train()
    optimizer.zero_grad()
    out = model_pt(X_tr_sc)
    loss = criterion(out, y_tr_t)
    loss.backward()
    optimizer.step()

model_pt.eval()
with torch.no_grad():
    preds = model_pt(X_te_sc).argmax(dim=1).numpy()
    acc = (preds == y_te_i).mean()
    print(f'PyTorch model accuracy: {acc:.4f}')

## Exercises

**Exercise 1:** Load a LFW face image and apply: grayscale, resize to 32×32, horizontal flip, and normalize to [0,1].

In [ ]:
# Exercise 1
from sklearn.datasets import fetch_lfw_people
from PIL import Image, ImageOps
import numpy as np
lfw = fetch_lfw_people(min_faces_per_person=70, resize=0.5)
# Apply preprocessing pipeline to lfw.images[0]

**Exercise 2:** Modify the CIFAR-10 CNN: add a third Conv2D block with 128 filters. Compare accuracy.

In [ ]:
# Exercise 2 — Add Conv2D block with 128 filters

**Exercise 3:** Create a PyTorch tensor of shape (3, 32, 32) representing an RGB image. Compute its mean per channel.

In [ ]:
# Exercise 3
import torch
img_tensor = torch.rand(3, 32, 32)  # random image
# Compute mean per channel (dim=1 and dim=2)

## Challenge Task

Improve the CIFAR-10 CNN:
1. Add **BatchNormalization** after each Conv2D layer
2. Add **Dropout(0.4)** after each MaxPooling layer
3. Use **learning rate scheduling** (ReduceLROnPlateau)
4. Train for 20 epochs
5. Report test accuracy and plot training curves

In [ ]:
# Challenge Task: Improved CIFAR-10 CNN
better_cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32,32,3)),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])
better_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Improved CNN parameters:', better_cnn.count_params())

##  Real-World AI/ML Application

| Application | Technique |
|---|---|
| Face recognition | CNN + embedding |
| Medical imaging | CNN for tumor detection |
| Document processing | Image preprocessing + OCR |
| Autonomous driving | CNN for object detection |
| Quality control | CNN for defect detection |

##  Day Summary

| Concept | Key Point |
|---|---|
| Image tensor | (H, W, C) array with values 0-1 |
| Pillow | Load, resize, convert, augment images |
| Conv2D | Filter slides across image — detects local patterns |
| MaxPooling | Reduces spatial dimensions, focuses on strongest features |
| CNN | Conv → Pool → Dense; standard for image classification |
| LFW dataset | Real face images; standard preprocessing: resize + normalize |
| PyTorch | Alternative DL framework with autograd |

**→ Day 10: NLP & OCR**